# CS514 NMF Test A: Within-Scope Taste-Dimension Validation

This notebook summarizes NMF Test A.

Question:

> Are the 11 manually selected user-profile dimensions recoverable directly from the ownership matrix, when the matrix is restricted to games inside those dimensions?

This test uses the merged ownership matrix and filters game columns to communities where `include_in_user_profiles = yes`. It then runs NMF with `k = 11` and compares NMF components to the 11 manual dimensions with cosine similarity and Hungarian matching.

The NMF run is produced by:

```text
scripts/cs514_nmf_test_a.py
```

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

BASE = ROOT / "data" / "processed" / "cs514_network_analysis"
OUT = BASE / "matrix_decomposition" / "nmf_test_a"
FIG = BASE / "figures" / "matrix_decomposition" / "nmf_test_a_similarity_heatmap.png"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

## 1. Diagnostics

In [ ]:
diagnostics = pd.read_csv(OUT / "nmf_test_a_diagnostics.csv")
diagnostics.T.rename(columns={0: "value"})

## 2. Manual Dimensions vs NMF Components

Higher cosine similarity means the NMF component places high weight on the same games contained in the manual dimension.

In [ ]:
best_matches = pd.read_csv(OUT / "nmf_test_a_best_matches.csv")
best_matches[[
    "manual_community",
    "manual_dimension",
    "nmf_component",
    "cosine_similarity",
    "manual_dimension_size",
    "component_top_game",
]].sort_values("cosine_similarity", ascending=False)

## 3. Similarity Heatmap

In [ ]:
display(Image(filename=str(FIG)))

## 4. Top Games Per NMF Component

In [ ]:
top_games = pd.read_csv(OUT / "nmf_test_a_top_games_by_component.csv")
for comp in sorted(top_games["nmf_component"].unique()):
    print(f"\nNMF component {comp}")
    display(top_games[top_games["nmf_component"] == comp][[
        "component_game_rank",
        "name",
        "overall_rank",
        "manual_community",
        "manual_dimension",
        "component_weight",
    ]].head(12))

## 5. Initial Readout

The current run gives:

- 8 / 11 matched dimensions with cosine similarity >= 0.50;
- 9 / 11 matched dimensions with cosine similarity >= 0.40;
- strongest matches for Golden Age canon, current heavy euros, historical wargames, Amerithrash/LCG, medium euros, dungeon crawl/campaign, puzzle/nature, and party/social;
- weak standalone recovery for Legacy + narrative mystery and Roll-and-write / number dice games.

This suggests that most large/stable profile dimensions are recoverable directly from the ownership matrix, while some smaller or supplementary dimensions may be better understood as cross-community additions rather than independent matrix factors.